<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Aprendizaje Automático Avanzado</h1>
    <h3>Ensemble Learning: Métodos de Ensamble</h3>
    
</div>


## ¿Qué es el Ensemble Learning?

El **aprendizaje por ensamble** (*ensemble learning*) es una técnica que **combina las predicciones de múltiples modelos** (llamados *estimadores base* o *aprendices débiles*) para producir un **modelo final más fuerte**.

> **Idea central — "la sabiduría de las multitudes":** así como un grupo diverso de personas suele tomar mejores decisiones que un solo individuo, un conjunto de modelos diversos suele predecir mejor que un modelo único.

Un **aprendiz débil** (*weak learner*) es un modelo apenas mejor que el azar (por ejemplo, un árbol de decisión poco profundo). Al combinar muchos, obtenemos un **aprendiz fuerte** (*strong learner*).

```mermaid
flowchart LR
    subgraph U["Modelo único"]
        direction LR
        D1[Datos] --> M1[Modelo] --> P1[Predicción]
    end
    subgraph E["Ensamble"]
        direction LR
        D2[Datos] --> M2a[Modelo 1]
        D2 --> M2b[Modelo 2]
        D2 --> M2c[Modelo 3]
        M2a --> C[Combinar<br/>voto o promedio]
        M2b --> C
        M2c --> C
        C --> P2[Predicción final]
    end
```

## ¿Por qué funcionan? El equilibrio sesgo–varianza

El error de un modelo puede descomponerse en tres partes:

$$\text{Error} = \text{Sesgo}^2 + \text{Varianza} + \text{ruido irreducible}$$

- **Sesgo (bias):** error por suposiciones demasiado simples (subajuste).
- **Varianza (variance):** sensibilidad excesiva a los datos de entrenamiento (sobreajuste).
- **Ruido irreducible:** aleatoriedad inherente a los datos.

Los métodos de ensamble atacan estas componentes de formas distintas:

| Método | Qué reduce principalmente | Cómo |
|---|---|---|
| **Bagging** | **Varianza** | Promedia modelos entrenados en muestras distintas |
| **Boosting** | **Sesgo** | Corrige secuencialmente los errores del modelo previo |
| **Stacking** | Ambos | Aprende cómo combinar modelos diversos |

>  **La clave es la diversidad:** el ensamble mejora solo si los modelos base **cometen errores diferentes**. Si todos se equivocan igual, combinarlos no aporta nada.

## Tipos de métodos de ensamble

Existen cuatro grandes familias:

```mermaid
flowchart TD
    EL([ENSEMBLE LEARNING])
    EL --> B[Bagging<br/>entrenamiento paralelo]
    EL --> Bo[Boosting<br/>entrenamiento secuencial]
    EL --> St[Stacking<br/>meta-modelo]
    EL --> V[Voting / Averaging]

    B --> RF[Random Forest]
    B --> ET[Extra Trees]
    Bo --> Ada[AdaBoost]
    Bo --> GB[Gradient Boosting<br/>XGBoost · LightGBM · CatBoost]
    St --> Meta[Meta-learner sobre<br/>las predicciones base]

    classDef root fill:#88dc65,color:#0000ff,stroke:#000047;
    class EL root;
```

| Familia | Estrategia | Modelos base | Ejemplos |
|---|---|---|---|
| **Bagging** | Paralela, muestras bootstrap | Independientes | Random Forest, Extra Trees |
| **Boosting** | Secuencial, corrige errores | Dependientes | AdaBoost, XGBoost, LightGBM, CatBoost |
| **Stacking** | Meta-modelo combina salidas | Heterogéneos | StackingClassifier |
| **Voting** | Voto / promedio directo | Heterogéneos | VotingClassifier |

## Bagging (Bootstrap Aggregating)

El **bagging** entrena muchos modelos **en paralelo**, cada uno sobre una **muestra bootstrap** distinta (muestreo con reemplazo) del dataset. Luego **agrega** sus predicciones: voto mayoritario (clasificación) o promedio (regresión).

```mermaid
flowchart TD
    D[Dataset original] --> S1[Muestra bootstrap 1]
    D --> S2[Muestra bootstrap 2]
    D --> S3[Muestra bootstrap N]
    S1 --> T1[Modelo 1]
    S2 --> T2[Modelo 2]
    S3 --> T3[Modelo N]
    T1 --> A[Agregar:<br/>voto mayoritario o promedio]
    T2 --> A
    T3 --> A
    A --> R[Predicción final]
```

### Random Forest
El **Random Forest** es el bagging más famoso: un ensamble de **árboles de decisión** con un truco extra — cada división considera solo un **subconjunto aleatorio de variables**, lo que aumenta la diversidad entre árboles.

- Reduce la **varianza** (combate el sobreajuste de árboles individuales).
- Robusto, poco sensible a hiperparámetros.
- Ofrece **importancia de variables**.

## Boosting

El **boosting** entrena los modelos **en secuencia**: cada nuevo modelo se enfoca en **corregir los errores** del anterior. La predicción final es una **suma ponderada** de todos.

```mermaid
flowchart LR
    D[Datos] --> M1[Modelo 1]
    M1 --> E1[Pondera<br/>errores]
    E1 --> M2[Modelo 2]
    M2 --> E2[Pondera<br/>errores]
    E2 --> M3[Modelo 3]
    M3 --> S[Suma ponderada]
    S --> R[Predicción final]
```

- **AdaBoost:** aumenta el **peso** de los ejemplos mal clasificados para que el siguiente modelo los priorice.
- **Gradient Boosting:** ajusta cada nuevo árbol a los **residuos** (gradiente del error) del modelo acumulado. Sus implementaciones **XGBoost, LightGBM y CatBoost** son el estado del arte en datos tabulares.

-  Reduce el **sesgo** (convierte aprendices débiles en uno muy preciso).
-  Pero es más propenso al sobreajuste que bagging: requiere ajustar `learning_rate` y número de árboles.


### Bagging vs. Boosting (de un vistazo)

```mermaid
flowchart LR
    subgraph BAG["Bagging - paralelo"]
        direction TB
        b1[Modelo 1]
        b2[Modelo 2]
        b3[Modelo 3]
    end
    subgraph BOO["Boosting - secuencial"]
        direction LR
        s1[Modelo 1] --> s2[Modelo 2] --> s3[Modelo 3]
    end
```

| | **Bagging** | **Boosting** |
|---|---|---|
| Entrenamiento | Paralelo (independiente) | Secuencial (dependiente) |
| Objetivo | Reducir **varianza** | Reducir **sesgo** |
| Datos por modelo | Muestra bootstrap | Todo, re-ponderado |
| Riesgo | Menor sobreajuste | Mayor sobreajuste |
| Ejemplo estrella | Random Forest | XGBoost / LightGBM / CatBoost |

## Voting y Averaging

El método más simple: entrenar **modelos heterogéneos** (por ejemplo, **regresión logística + SVM + k-NN**) y combinar sus salidas.

- **Hard voting:** gana la clase más votada.
- **Soft voting:** promedia las **probabilidades** y elige la mayor (suele funcionar mejor).
- **Averaging:** promedio de predicciones (para regresión).

``` Python
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

vote = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=5000)),
        ('svc', SVC(probability=True)),
        ('knn', KNeighborsClassifier()),
    ],
    voting='soft',
).fit(X_train, y_train)

print('Voting (soft) ->', round(accuracy_score(y_test, vote.predict(X_test)), 4))
```

## Stacking (Stacked Generalization)

El **stacking** es el método más sofisticado: entrena varios **modelos base heterogéneos** y luego un **meta-modelo** aprende la **mejor forma de combinar** sus predicciones.

```mermaid
flowchart TD
    D[Datos] --> B1[Modelo base 1<br/>Random Forest]
    D --> B2[Modelo base 2<br/>Gradient Boosting]
    D --> B3[Modelo base 3<br/>SVM]
    B1 --> Meta[Meta-modelo<br/>Regresión logística]
    B2 --> Meta
    B3 --> Meta
    Meta --> R[Predicción final]
```

En lugar de reglas fijas (como el voto), el **meta-modelo aprende** cuánto confiar en cada modelo base según sus predicciones.

``` Python
from sklearn.ensemble import StackingClassifier

stack = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('svc', SVC(probability=True)),
    ],
    final_estimator=LogisticRegression(max_iter=5000),
    cv=5,
).fit(X_train, y_train)

print('Stacking ->', round(accuracy_score(y_test, stack.predict(X_test)), 4))

## En Resumen

- El **Ensemble Learning** combina varios modelos para superar a cualquiera individual ("sabiduría de las multitudes").
- Funciona gracias al equilibrio **sesgo–varianza** y a la **diversidad** entre modelos.
- **Bagging** (Random Forest) reduce la **varianza**; **Boosting** (XGBoost, LightGBM, CatBoost) reduce el **sesgo**.
- **Voting** combina modelos heterogéneos por voto/promedio; **Stacking** entrena un **meta-modelo** que aprende a combinarlos.
- Son **algoritmos avanzados** y el **estado del arte para datos tabulares**.


[scikit-learn — Ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html)
